# Monte Carlo: Coherent vs Fock

This quick notebook highlights why the Monte Carlo coherent-state result looks almost deterministic, while the Fock-state result shows visible finite-trajectory sampling noise.

Current project assumptions used here:
- single site
- dissipation only
- `interaction_strength = 0`
- Monte Carlo remains trajectory-based even when `U = 0`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science", "no-latex"])
except ImportError:
    pass

plt.rcParams["text.usetex"] = False
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
    }
)

PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"

paths = {
    "exact_coherent": OUTPUT_DIR / "exact_coherent_numOfParticles1_gamma1_time5_dt0p001_numOfSamples1.csv",
    "mc_coherent": OUTPUT_DIR / "monteCarlo_coherent_numOfParticles1_gamma1_time5_dt0p001_numOfSamples1000_hilbertSize6.csv",
    "exact_fock": OUTPUT_DIR / "exact_fock_numOfParticles1_gamma1_time5_dt0p001_numOfSamples1.csv",
    "mc_fock": OUTPUT_DIR / "monteCarlo_fock_numOfParticles1_gamma1_time5_dt0p001_numOfSamples1000_hilbertSize2.csv",
}

for label, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file for {label}: {path}")

datasets = {label: pd.read_csv(path) for label, path in paths.items()}
datasets["exact_coherent"].head()

In [ ]:
summary = pd.DataFrame(
    {
        "state": ["coherent", "fock"],
        "max_abs_mean_error": [
            np.max(np.abs(datasets["mc_coherent"]["mean_particle_number"] - datasets["exact_coherent"]["mean_particle_number"])),
            np.max(np.abs(datasets["mc_fock"]["mean_particle_number"] - datasets["exact_fock"]["mean_particle_number"])),
        ],
        "max_abs_variance_error": [
            np.max(np.abs(datasets["mc_coherent"]["variance"] - datasets["exact_coherent"]["variance"])),
            np.max(np.abs(datasets["mc_fock"]["variance"] - datasets["exact_fock"]["variance"])),
        ],
    }
)
summary

## Full Comparison

The coherent Monte Carlo curve nearly lies on top of the exact curve, while the Fock Monte Carlo curve shows visible stair-stepping from finite trajectory averaging.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex="col")

plot_specs = [
    ("coherent", "mean_particle_number", "Mean Particle Number"),
    ("fock", "mean_particle_number", "Mean Particle Number"),
    ("coherent", "variance", "Variance"),
    ("fock", "variance", "Variance"),
]

for ax, (state, column, ylabel) in zip(axes.flatten(), plot_specs):
    exact = datasets[f"exact_{state}"]
    mc = datasets[f"mc_{state}"]
    ax.plot(exact["time"], exact[column], color="black", linewidth=2.0, label="Exact")
    ax.plot(mc["time"], mc[column], color="tab:orange", linewidth=1.5, label="Monte Carlo")
    ax.set_title(f"{state.capitalize()} | {ylabel}")
    ax.set_xlabel("Time")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

axes[0, 0].legend()
axes[0, 1].legend()
fig.tight_layout()
plt.show()

## Difference From Exact

Looking at `Monte Carlo - Exact` makes the contrast much clearer.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for row, state in enumerate(["coherent", "fock"]):
    exact = datasets[f"exact_{state}"]
    mc = datasets[f"mc_{state}"]
    mean_diff = mc["mean_particle_number"] - exact["mean_particle_number"]
    variance_diff = mc["variance"] - exact["variance"]

    axes[row, 0].plot(mc["time"], mean_diff, color="tab:blue", linewidth=1.4)
    axes[row, 0].axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
    axes[row, 0].set_title(f"{state.capitalize()} | Mean Difference")
    axes[row, 0].set_ylabel("MC - Exact")
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].plot(mc["time"], variance_diff, color="tab:green", linewidth=1.4)
    axes[row, 1].axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
    axes[row, 1].set_title(f"{state.capitalize()} | Variance Difference")
    axes[row, 1].grid(True, alpha=0.3)

for ax in axes[-1, :]:
    ax.set_xlabel("Time")

fig.tight_layout()
plt.show()

## Short Interpretation

For the current pure-loss model, the jump operator is proportional to `a`.

For a coherent state:
- `a |alpha> = alpha |alpha>`
- the coherent state is an eigenstate of the jump operator
- after a jump and renormalization, the trajectory stays on the same coherent-state manifold
- so the ensemble-averaged observables look nearly deterministic

For a Fock state:
- `a |n> = sqrt(n) |n-1>`
- each jump changes the occupation number discretely
- different trajectories accumulate different jump histories
- finite trajectory averaging then shows visible stochastic sampling noise

So the Monte Carlo solver is stochastic in both cases, but the coherent initial state hides that stochasticity much more strongly in the saved observables for this specific `U = 0`, dissipation-only problem.

## Optional Next Step

If you want the contrast to be even stronger, a useful follow-up test is to compare smaller `ntraj` values such as 50, 100, and 250 for both initial states. The Fock-state Monte Carlo error should become visibly worse much faster than the coherent-state error.